# 3. Frames to OpenCV, PyTorch and TensorFlow

`frame.image` is a NumPy array (height x width x 3, RGB, `uint8`), so it goes straight into most image tools.

OpenCV comes with the notebook tools. PyTorch and TensorFlow are not in any extra; install the one you want yourself, for example `pip install torch` or `pip install tensorflow`. Sections for tools you have not installed are skipped.

Log in first with [1. Quick start](01-quick-start.ipynb).

In [ ]:
from importlib.util import find_spec

import googlenestcam as gnc

cam = gnc.list_cameras()[0]
frame = cam.snapshot(size=480)
print(frame.image.shape)

## OpenCV

OpenCV uses BGR order, so swap the colours first.

In [ ]:
import cv2

bgr = cv2.cvtColor(frame.image, cv2.COLOR_RGB2BGR)
edges = cv2.Canny(bgr, 100, 200)
print(edges.shape)

## PyTorch

Most models want channels first and values from 0 to 1.

In [ ]:
if find_spec("torch") is None:
    print("PyTorch is not installed; skipping.")
else:
    import torch

    tensor = torch.from_numpy(frame.image).permute(2, 0, 1).float() / 255
    print(tensor.shape)

## TensorFlow

TensorFlow takes the array as it is: height x width x channels.

In [ ]:
if find_spec("tensorflow") is None:
    print("TensorFlow is not installed; skipping.")
else:
    import tensorflow as tf

    tensor = tf.convert_to_tensor(frame.image, dtype=tf.float32) / 255
    print(tensor.shape)

## Many Frames

For a model that runs on every Frame, use a Stream instead of many Snapshots. `frames="all"` keeps every Frame in a queue, so none are missed.

In [ ]:
with cam.stream(frames="all", size=480, audio=False) as stream:
    for count, frame in enumerate(stream.frames()):
        bgr = cv2.cvtColor(frame.image, cv2.COLOR_RGB2BGR)
        print(frame.time, cv2.mean(bgr)[:3])
        if count == 10:
            break